# 09 — XUV instrument response

Applies the paper's eq. 1 (Al filter × QE × grating × Al₂O₃ × CH) to a
simulated harmonic spectrum.  We also inspect the grating higher-order
diffraction logistic (eq. 6) that makes low-order harmonics see leakage
from harmonics 2n and 3n.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from harmonyemissions import Laser, Target, simulate
from harmonyemissions.config import NumericsConfig
from harmonyemissions.detector.deconvolve import DetectorConfig, apply_instrument_response, spectral_response
from harmonyemissions.detector.grating import grating_order_ratio
from harmonyemissions.detector.al_filter import harmonic_to_wavelength_nm

In [ ]:
numerics = NumericsConfig(pipeline_grid=96, pipeline_dx_um=0.1,
                          diag_harmonics=(1, 20, 40))
result = simulate(Laser(a0=24.0, spatial_profile='super_gaussian', spot_fwhm_um=2.0,
                        angle_deg=45.0),
                  Target.sio2(t_HDR_fs=351.0),
                  model='surface_pipeline', numerics=numerics)
ccd = apply_instrument_response(result.spectrum, 0.8, DetectorConfig(al_thickness_um=1.5))
result.instrument_spectrum = ccd

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
n = result.spectrum.coords['harmonic'].values
axes[0].loglog(n, result.spectrum.values / result.spectrum.values.max(), label='simulation (normalised)')
axes[0].loglog(n, ccd.values / max(ccd.values.max(), 1e-30), label='CCD after S(λ)')
axes[0].set_xlabel('harmonic order n')
axes[0].set_ylabel('normalised signal')
axes[0].set_xlim(1, 80)
axes[0].legend()
axes[0].set_title('Instrument response vs raw simulation')

ns = np.arange(5, 80)
axes[1].plot(ns, grating_order_ratio(ns, 2), label='2nd order (eq. 6, paper)')
axes[1].plot(ns, grating_order_ratio(ns, 3), label='3rd order')
axes[1].set_xlabel('harmonic order n')
axes[1].set_ylabel('R_{s_m/s_1}(n)')
axes[1].set_title('Grating higher-order logistic fits')
axes[1].legend()
fig.tight_layout()
plt.show()